In [1]:
import sys, os, yaml
import pandas as pd
sys.path.append(os.path.abspath(os.path.join('..')))

from src.models.supervised import prepare_classification_data, train_classifiers
from src.visualization.plots import plot_confusion_matrix
from src.models.handler import save_model_artifact
from sklearn.metrics import classification_report

# 1. Load Data & Config
with open('../configs/params.yaml', 'r') as f:
    config = yaml.safe_load(f)

df = pd.read_csv(f"../{config['processed_data_path']}")



In [ ]:
# 2. Split Data


(X_train, X_test, y_train, y_test), le_target = prepare_classification_data(df, config)

In [ ]:
# 3. Train Models
rf_model, xgb_model = train_classifiers(X_train, y_train, config)


In [ ]:
# 4. Đánh giá F1-macro 
y_pred_rf = rf_model.predict(X_test)
print("--- KẾT QUẢ RANDOM FOREST ---")

print(classification_report(y_test, y_pred_rf, target_names=le_target.classes_))

--- KẾT QUẢ RANDOM FOREST ---
              precision    recall  f1-score   support

        rain       1.00      1.00      1.00     17101
        snow       1.00      1.00      1.00      2190

    accuracy                           1.00     19291
   macro avg       1.00      1.00      1.00     19291
weighted avg       1.00      1.00      1.00     19291



In [ ]:
# 5. Phân tích lỗi giao mùa/cực trị (Yêu cầu bài toán)
# Lọc những dòng dự báo sai
results = X_test.copy()
results['Actual'] = le_target.inverse_transform(y_test)
results['Predicted'] = le_target.inverse_transform(y_pred_rf)
errors = results[results['Actual'] != results['Predicted']]

print(f"Số lượng lỗi: {len(errors)}")
# Xem lỗi tập trung vào tháng nào (Giao mùa: tháng 3, 4 hoặc 10, 11)
print("Thống kê lỗi theo tháng:")
print(errors['Month'].value_counts().sort_index())



Số lượng lỗi: 0
Thống kê lỗi theo tháng:
Series([], Name: count, dtype: int64)


In [5]:
# 6. Lưu mô hình
save_model_artifact(rf_model, "rf_weather_model", config)
save_model_artifact(xgb_model, "xgb_weather_model", config)

📦 Đã lưu mô hình: ../outputs/models/rf_weather_model.pkl
📦 Đã lưu mô hình: ../outputs/models/xgb_weather_model.pkl
